# ConvNeXt-Tiny: 2단계 선별(v3.2) × 질감 마스크 / 베드 지도 마스크 — 3시드 × 5fold

## 왜 돌리나
DINOv2 격자(빠른 모델)에서 "선별을 하고 나면 성능 차이는 주로 마스크에서 난다"는 신호가 나왔다 [확인: 빠른 모델 1회].
이게 본 모델(ConvNeXt)에서도 같은지 확인한다. 이미 3시드 결과가 있는 **재활용×재활용**, **질감×질감** 과 같은 조건으로 두 조합을 추가한다.

| 조건 | 학습 사진 (선별) | 입력 마스크 |
|---|---|---|
| v4_s2_texture | 2단계 v3.2 통과 (팀원 reference.json 그대로) | 질감 마스크 (기존 질감 실험과 같은 파일) |
| v5_s2_bedmap | 2단계 v3.2 통과 | 베드 지도 마스크 (2단계 선별의 칸 판정 + 직선 보정) |

- 선별은 같고 마스크만 다르다 → 두 조건의 차이 = **마스크 효과**.
- v4와 기존 "질감×질감"은 마스크가 같고 선별만 다르다 → 차이 = **선별 효과** (공통 사진에서 비교).

## 옛 실험과 같게 맞춘 것 (그래야 기존 3시드 결과와 나란히 놓을 수 있다)
- 엔진: 2026-10-05 `engine_convnext.py` 복사본. 바뀐 곳은 조건 이름·어댑터 파일명 두 줄뿐.
- 입력 512, 배치 16, 20라운드, AdamW(몸통 1e-4 / 분류층 1e-3), cosine, 같은 증강, AMP, 마지막 가중치, 기준 0.5.
- **업데이트 횟수:** fold마다 옛 v1 학습 장수 기준(예: fold0 = 2,840번). 학습 사진 수가 달라도 "공부 시간"을 똑같이 준다.
  - 왜: 사진이 많은 조건이 더 오래 학습해서 좋아 보이는 걸 막는다. 회계로 치면 부서별 실적을 같은 기간(12개월)으로 비교하는 것.
- 분할: 같은 시간 블록 5-fold + buffer 30프레임 (`folds.csv`, 해시로 확인).

## 옛 실험과 다른 것
- 평가 사진: 각 fold의 평가 구간 **전부**(5fold 합쳐 4,000장). 선별 밖 사진도 점수를 낸다. DINOv2 격자와 같은 방식이다.
- 면적 10% 제외 없음 (격자와 같은 규칙). 질감 마스크가 거의 빈 사진이 fold당 1~8장 학습에 들어간다. 면적은 기록한다.

## 한계
클래스별 출력 1회, 독립 테스트 아님, 2단계 판정 영역(위쪽 85%)과 모델 입력(고정 크롭)이 다름, 회색 면적 지름길 미해결.

## 1. 설정
Drive `산공종설/`에 아래 5개가 있어야 한다. **새로 올릴 건 `convnext_stage2v32_patch_20261013.zip` 하나뿐**이다.

| 파일 | 내용 | 상태 |
|---|---|---|
| underfill_originals_4000.zip | 원본 4,000장 | 이미 있음 |
| resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip | 공통 분류 함수, folds.csv, 예산 기준 목록 | 이미 있음 |
| convnext_three_preproc_patch_20261005.zip | 질감 마스크 4,000장 | 이미 있음 |
| dinov2_grid_bundle_20261013.zip | 2단계 v3.2 목록, 베드 지도 마스크 4,000장 | 이미 있음 |
| convnext_stage2v32_patch_20261013.zip | 이번 어댑터 + 엔진 복사본 | **새로 올림** |

런타임은 L4 GPU. 끊겨도 같은 RUN_NAME으로 위부터 다시 실행하면 이어서 한다 (라운드마다 저장).

In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
ORIGINAL_ZIP_NAME = 'underfill_originals_4000.zip'
BUNDLE_ZIP_NAME = 'resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip'
TEXTURE_ZIP_NAME = 'convnext_three_preproc_patch_20261005.zip'
GRID_ZIP_NAME = 'dinov2_grid_bundle_20261013.zip'
PATCH_ZIP_NAME = 'convnext_stage2v32_patch_20261013.zip'
RUN_NAME = 'convnext_stage2v32_20261013_run01'
IMG_SIZE = 512
BATCH = 16
ROUNDS = 20
SEEDS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]
RUN_TRAINING = True         # False면 입력 준비·점검만
RUN_REMAINING_SEEDS = True  # 시드0만 보고 멈추려면 False

In [ ]:
import torch, torchvision
from google.colab import drive
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name())
drive.mount('/content/drive')
OUTPUT = DRIVE_DIR / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)

## 2. 파일 풀기·무결성 점검
ZIP을 런타임 SSD(`/content`)에 푼다. Drive에서 바로 읽으면 느리다.
`--verify-only`는 학습 없이 10가지(2조건×5fold) 분할이 buffer 30프레임을 지키는지, 업데이트 횟수가 옛 실험과 같은지만 확인한다.
- 확인할 것: `eligibility_audit.csv`의 `total_update_steps`가 fold0~4에서 2840/2460/2260/2580/2740 (옛 실험과 동일)

In [ ]:
import zipfile, hashlib, json, shutil, subprocess, sys, os
import pandas as pd
from IPython.display import display
LOCAL = Path('/content/convnext_stage2v32_20261013')
LOCAL.mkdir(parents=True, exist_ok=True)
BUNDLE, ORIGINALS, TEXPATCH, GRID, PATCH = (LOCAL/n for n in ('bundle', 'originals', 'texpatch', 'grid', 'patch'))
CACHE = LOCAL / f'cache_{IMG_SIZE}'

def extract_zip(source, target):
    assert source.is_file(), f'파일 없음: {source}'
    local_zip = LOCAL / source.name
    if not local_zip.exists() or local_zip.stat().st_size != source.stat().st_size:
        shutil.copy2(source, local_zip)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for item in archive.infolist():
            assert (target/item.filename).resolve().is_relative_to(target.resolve()), 'ZIP 경로 오류'
        archive.extractall(target)
    print('준비:', source.name)

for name, target in ((BUNDLE_ZIP_NAME, BUNDLE), (TEXTURE_ZIP_NAME, TEXPATCH), (GRID_ZIP_NAME, GRID),
                     (PATCH_ZIP_NAME, PATCH), (ORIGINAL_ZIP_NAME, ORIGINALS)):
    extract_zip(DRIVE_DIR / name, target)
for folder in (BUNDLE, TEXPATCH, PATCH):
    hashes = json.loads((folder/'checksums.json').read_text())
    for name, expected in hashes.items():
        assert hashlib.sha256((folder/name).read_bytes()).hexdigest() == expected, name
    print('해시 일치:', folder.name, len(hashes), '개')
os.environ['PYTHONPATH'] = str(PATCH) + os.pathsep + str(BUNDLE) + os.pathsep + os.environ.get('PYTHONPATH', '')
BASE_COMMAND = [sys.executable, '-u', str(PATCH/'train_convnext_stage2v32.py'),
    '--bundle', str(BUNDLE), '--texture', str(TEXPATCH/'texture'), '--grid', str(GRID),
    '--originals', str(ORIGINALS), '--cache', str(CACHE), '--output', str(OUTPUT),
    '--size', str(IMG_SIZE), '--batch', str(BATCH), '--rounds', str(ROUNDS)]
subprocess.run(BASE_COMMAND + ['--verify-only'], check=True)
audit = pd.read_csv(OUTPUT/'eligibility_audit.csv')
display(audit)
assert audit.groupby('fold').total_update_steps.first().tolist() == [2840, 2460, 2260, 2580, 2740]
display(pd.read_csv(OUTPUT/'coverage.csv'))

## 3. 입력 캐시 만들기 + 눈으로 확인
4,000장 × 2벌(질감 / 베드 지도)을 512×512로 줄여 SSD에 저장한다 (약 6GB, 10~15분).
- 왜 미리 만드나: 20라운드 동안 같은 사진을 수십 번 읽는데, 매번 JPG를 풀고 마스크를 칠하면 GPU가 놀게 된다.
- 마스크 밖은 회색 RGB(124,116,104)로 칠한다. 기존 실험과 같은 색이다.
- 아래 그림에서 확인할 것: 질감 마스크와 베드 지도가 같은 사진에서 어떻게 다르게 남기는지. 베드가 보이는 사진(아래 두 줄)에서 차이가 크게 나야 정상이다.

In [ ]:
# 캐시 표시 파일(complete.json)이 이미 있으면 다시 만들지 않는다 (아래 6번 설명 참고)
if not (CACHE/'complete.json').exists():
    subprocess.run(BASE_COMMAND + ['--prepare-only'], check=True)
import numpy as np
import matplotlib.pyplot as plt
inputs = pd.read_csv(OUTPUT/'input_manifest.csv')
arrays = {n: np.load(CACHE/f'{n}.npy', mmap_mode='r') for n in ('texture', 'bedmap')}
picks = []
for clean in (1, 0):
    for label in ('normal', 'underfill_50'):
        g = inputs[(inputs.common == 1) & (inputs.clean_old == clean) & (inputs.label == label)]
        picks.append((g.index[len(g)//2], 'clean' if clean else 'bed visible'))
fig, axes = plt.subplots(len(picks), 2, figsize=(8, 4*len(picks)), squeeze=False)
for i, (idx, kind) in enumerate(picks):
    r = inputs.iloc[idx]
    for j, n in enumerate(('texture', 'bedmap')):
        axes[i, j].imshow(arrays[n][idx].transpose(1, 2, 0)); axes[i, j].axis('off')
        axes[i, j].set_title(f'{n}: {r.label} #{r.frame_index} ({kind})')
fig.tight_layout(); plt.show(); plt.close(fig)

## 4. 실행·결과 표시 함수
긴 로그는 Drive의 `train_seed*.log`에 저장하고 화면에는 fold 완료 줄만 띄운다.
결과 표에서 볼 것 (순위 비교는 `common` 3,039장에서만):
- `common`: 세 선별 모두 통과한 공통 사진. DINOv2 격자 A와 같은 사진이다.
- `common_clean` / `common_bed_visible`: 공통 사진을 깨끗한 사진 2,648 / 베드가 보이는 사진 391로 나눔. **마스크 차이는 베드가 보이는 쪽에서 나야 한다** [추정: 격자 결과 기반].
- `own_stage2`: 2단계가 판정한 3,062장 (운영 성능).
- `none_pass`: 세 선별 모두 보류한 539장. 정답률이 낮아도 정상이다 (판정 불가 사진이라 선별이 필요한 근거).

In [ ]:
def export_results(suffix):
    destination = DRIVE_DIR / f'{RUN_NAME}_{suffix}_results.zip'
    temp = LOCAL / destination.name
    with zipfile.ZipFile(temp, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix.lower() in {'.csv', '.json', '.log', '.png', '.md'}:
                archive.write(path, path.relative_to(OUTPUT).as_posix())
    shutil.copy2(temp, destination)
    print('공유할 결과 ZIP:', destination)
    return destination

def show_results(seed):
    progress = json.loads((OUTPUT/'progress.json').read_text())
    assert seed in progress['fully_completed_seeds'], '이 시드의 10회가 아직 완료되지 않았습니다.'
    print(f'시드 {seed} 완료 — 전체 {progress["completed_runs"]}/30회')
    s = pd.read_csv(OUTPUT/'summary.csv')
    display(s[s.seed == seed][['version', 'group', 'n', 'accuracy', 'underfill_recall', 'normal_recall', 'auroc', 'fp', 'fn']])
    p = pd.read_csv(OUTPUT/'paired_comparison.csv')
    print('같은 사진에서 한쪽만 맞힌 수 (a=질감, b=베드 지도)'); display(p[p.seed == seed])
    m = pd.read_csv(OUTPUT/'across_seeds.csv')
    display(m[(m.group == 'common') & m.metric.isin(['accuracy', 'fp', 'fn', 'auroc'])])
    return export_results(f'seed{seed}')

def train_seed(seed):
    progress = OUTPUT/'progress.json'
    if progress.exists() and seed in json.loads(progress.read_text())['fully_completed_seeds']:
        print(f'시드 {seed}는 이미 10회 완료 — 학습 없이 결과만 다시 표시합니다.')
        return show_results(seed)
    # 어댑터의 캐시 확인 코드에 버그가 있다: 회색값을 튜플로 비교하는데 저장된 JSON에서는 리스트로 읽혀 항상 "설정 다름"이 된다.
    # 그래서 학습 명령마다 표시 파일을 지워 캐시를 새로 만든다 (시드당 10~15분 추가). 같은 코드로 같은 입력이 다시 만들어지므로 결과에는 영향이 없다.
    # 어댑터를 고치면 코드 지문(sha)이 달라져 시드0과 기록이 갈라지므로, 코드는 그대로 두고 이 방법을 쓴다.
    (CACHE/'complete.json').unlink(missing_ok=True)
    log_path = OUTPUT / f'train_seed{seed}.log'
    command = BASE_COMMAND + ['--seeds', str(seed), '--folds', *map(str, FOLDS)]
    print(f'시드 {seed}: 2조건 × 5fold = 10회 시작. 로그: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as logfile:
        logfile.write('\n실행 명령: '+json.dumps(command, ensure_ascii=False)+'\n')
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            logfile.write(line); logfile.flush()
            if f'round {ROUNDS}/{ROUNDS}:' in line or '이미 완료' in line:
                print(line.rstrip(), flush=True)
        return_code = process.wait()
    if return_code:
        print(log_path.read_text(encoding='utf-8')[-7000:])
        raise RuntimeError(f'학습 오류. 기존 저장은 유지됩니다. 로그: {log_path}')
    return show_results(seed)

## 5. 시드 0 먼저 (약 2시간, L4 기준)
옛 실험은 L4에서 fold당 약 12분이었다 [확인: 옛 결과 complete.json]. 이번엔 10회 → 약 2시간 + 캐시 시간.
끝나면 표와 `..._seed0_results.zip`이 나온다. 이것만 먼저 공유해도 된다.

In [ ]:
assert SEEDS == [0, 1, 2] and FOLDS == [0, 1, 2, 3, 4]
if RUN_TRAINING:
    SEED0_ZIP = train_seed(0)
else:
    print('입력 준비만 완료했습니다. 학습하지 않았습니다.')

## 6. 시드 1·2 (약 4시간)
시드마다 ZIP을 따로 남기므로 중간에 끊겨도 앞 결과는 보존된다. 실험 중 설정·묶음을 바꾸지 말 것.
- 왜 3시드: ConvNeXt는 초기값·사진 순서에 따라 결과가 흔들린다. 1시드 차이가 시드 간 흔들림보다 작으면 "차이 있다"고 말할 수 없다.

In [ ]:
if RUN_TRAINING and RUN_REMAINING_SEEDS:
    for seed in SEEDS[1:]:
        train_seed(seed)
    print('3시드 × 2조건 × 5fold = 30회 완료')
    FINAL_ZIP = export_results('all3seeds')
elif RUN_TRAINING:
    print('시드0 결과만 보관하고 멈췄습니다.')

## 7. 공유
`..._seed0_results.zip`(중간), `..._all3seeds_results.zip`(최종)을 받아 오면 된다. 모델 가중치(약 3GB)는 Drive 결과 폴더에만 남는다.
기존 재활용×재활용·질감×질감 3시드 결과와의 비교는 로컬에서 공통 3,039장 기준으로 따로 맞춰 계산한다 (옛 실험은 면적 10% 제외가 있어서 평가 사진 목록이 조금 다르다).

## 8. 런타임 반납
결과 ZIP이 Drive에 있는지 확인한 뒤 실행. 학습 중에는 실행하지 말 것.

In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()